<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/ROSG_Test6e_i5_stochastic_heat_trace_DSI_transmission_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROSG Test6e — i=5 stochastic heat-trace DSI transmission audit

**Mode:** Codex + autotests + stochastic heat trace.

This notebook extends the Test6d V2 i=3→i=4 result to \(i=5\) using Hutchinson stochastic trace estimation and `expm_multiply`.

It tests whether an explicitly imposed DSI modulation at link/conductance level is transmitted through \(L_{5,Z}\), \(P_5(u;Z)\), and \(d_s^{eff}(Z)\). It does **not** prove spontaneous DSI.

## Imports, configuration and scenarios

In [ ]:
# ============================================================
# ROSG Test6e — i=5 stochastic heat-trace DSI transmission audit
# ------------------------------------------------------------
# Purpose:
#   Extend Test6d V2 from i=3,4 to i=5 without exact diagonalization.
#
#   Uses Hutchinson stochastic trace estimation + scipy.sparse.linalg.expm_multiply:
#
#       P_i(u;Z) = (1/N) Tr(exp(-u L_{i,Z}))
#
#   with N = |V_5| = 1089.
#
# Tested scenarios:
#   H0_smooth
#   H1_lattice_coherent_A002
#   CTRL_nonlattice_coherent_A002
#   CTRL_lattice_randomphase_A002
#
# Important:
#   This is not a proof of spontaneous DSI.
#   It tests whether an explicitly imposed link-level DSI modulation
#   remains transmitted at i=5.
# ============================================================

import os, json, math, shutil, zipfile, hashlib, time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import scipy.sparse as sp
    from scipy.sparse.linalg import expm_multiply, eigsh
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False


@dataclass
class Scenario:
    name: str
    amplitude: float
    omega_mode: str = "lattice"       # lattice, nonlattice
    phase_mode: str = "coherent"      # coherent, random
    phase_value: float = 0.0


@dataclass
class Test6eConfig:
    project_name: str = "ROSG_Test6e_i5_stochastic_heat_trace_DSI_transmission_audit"

    level_i: int = 5
    lambda_lattice: float = 2.0

    # Dense enough for omega0=2*pi/log(2)=9.0647:
    # RUN_FAST: 33 points -> dZ=0.1875, Nyquist≈16.76 > omega0
    # publication: 65 points -> dZ=0.09375, Nyquist≈33.51
    RUN_FAST: bool = False
    Z_min: float = -1.0
    Z_max: float = 5.0
    n_Z_fast: int = 33
    n_Z_publication: int = 65

    u_min: float = 0.03
    u_max: float = 40.0
    n_u_fast: int = 22
    n_u_publication: int = 34

    n_probes_fast: int = 8
    n_probes_publication: int = 24

    base_conductance: float = 1.0
    vertical_gain: float = 2.1
    activation_center_Z: float = 1.75
    activation_width_Z: float = 0.75

    baseline_poly_degree: int = 3
    permutation_count_fast: int = 250
    permutation_count_publication: int = 600

    random_seed: int = 20260605

    out_dir: str = "/content/ROSG_Test6e_i5_stochastic_heat_trace_DSI_transmission_audit"
    use_google_drive: bool = True
    drive_dir: str = "/content/drive/MyDrive/ROSG_exports/ROSG_Test6e_i5_stochastic_heat_trace_DSI_transmission_audit"

CFG = Test6eConfig()

SCENARIOS = [
    Scenario("H0_smooth", 0.0, "lattice", "coherent", 0.0),
    Scenario("H1_lattice_coherent_A002", 0.02, "lattice", "coherent", 0.0),
    Scenario("CTRL_nonlattice_coherent_A002", 0.02, "nonlattice", "coherent", 0.0),
    Scenario("CTRL_lattice_randomphase_A002", 0.02, "lattice", "random", 0.0),
]


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def prepare_output_dir(cfg=CFG):
    out = Path(cfg.out_dir)
    if cfg.use_google_drive and in_colab():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive", force_remount=False)
            if Path("/content/drive/MyDrive").exists():
                out = Path(cfg.drive_dir)
        except Exception as exc:
            print("[Drive warning]", exc)
            out = Path(cfg.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    return out


OUT_DIR = prepare_output_dir(CFG)
CACHE_DIR = OUT_DIR / "_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)
print("RUN_FAST:", CFG.RUN_FAST)


def active_params(cfg=CFG):
    return {
        "n_Z": cfg.n_Z_fast if cfg.RUN_FAST else cfg.n_Z_publication,
        "n_u": cfg.n_u_fast if cfg.RUN_FAST else cfg.n_u_publication,
        "n_probes": cfg.n_probes_fast if cfg.RUN_FAST else cfg.n_probes_publication,
        "permutation_count": cfg.permutation_count_fast if cfg.RUN_FAST else cfg.permutation_count_publication,
    }

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directory: /content/drive/MyDrive/ROSG_exports/ROSG_Test6e_i5_stochastic_heat_trace_DSI_transmission_audit
RUN_FAST: False


## IFS/PCF graph and modulated conductance model

In [ ]:
# ============================================================
# IFS/PCF graph and modulated conductance model
# ============================================================

def omega0_lattice(cfg=CFG):
    return float(2.0 * math.pi / math.log(cfg.lambda_lattice))


def omega_for_scenario(scenario, cfg=CFG):
    omega0 = omega0_lattice(cfg)
    if scenario.omega_mode == "lattice":
        return omega0
    if scenario.omega_mode == "nonlattice":
        return float(omega0 * math.sqrt(2.0))
    raise ValueError(f"Unknown omega_mode: {scenario.omega_mode}")


def grid_size_from_i(i):
    return 2 ** int(i)


def num_points_i(i):
    L = grid_size_from_i(i)
    return int((L + 1) ** 2)


def build_Ei(i):
    L = grid_size_from_i(i)
    rows = []
    eid = 0

    def node_id(x, y):
        return y * (L + 1) + x

    for y in range(L + 1):
        for x in range(L + 1):
            u = node_id(x, y)
            if x < L:
                rows.append({"edge_id": eid, "i": int(i), "u": int(u), "v": int(node_id(x+1, y)), "orientation": "horizontal"})
                eid += 1
            if y < L:
                rows.append({"edge_id": eid, "i": int(i), "u": int(u), "v": int(node_id(x, y+1)), "orientation": "vertical"})
                eid += 1
    return pd.DataFrame(rows)


def local_hierarchy_level_for_edge(row, i):
    L = grid_size_from_i(i)
    u, v = int(row["u"]), int(row["v"])
    ux, uy = u % (L + 1), u // (L + 1)
    vx, vy = v % (L + 1), v // (L + 1)
    mx2 = ux + vx
    my2 = uy + vy

    def trailing_zeros(a):
        a = int(abs(a))
        if a == 0:
            return int(i)
        c = 0
        while a % 2 == 0 and c < i:
            c += 1
            a //= 2
        return c

    return int(min(i, max(trailing_zeros(mx2), trailing_zeros(my2))))


def prepare_edge_template(i, cfg=CFG):
    Ei = build_Ei(i)
    hs, thetas = [], []
    for _, row in Ei.iterrows():
        h = local_hierarchy_level_for_edge(row, i)
        theta = cfg.activation_center_Z - 0.12 * h
        hs.append(h)
        thetas.append(theta)
    Ei["hierarchy_h"] = hs
    Ei["theta_edge"] = thetas
    return Ei


def edge_phases(Ei, scenario, i, cfg=CFG):
    if scenario.phase_mode == "coherent":
        return np.zeros(len(Ei), dtype=float) + float(scenario.phase_value)
    if scenario.phase_mode == "random":
        rng = np.random.default_rng(cfg.random_seed + 7919 * int(i) + len(scenario.name))
        return rng.uniform(0.0, 2.0 * math.pi, size=len(Ei))
    raise ValueError(f"Unknown phase_mode: {scenario.phase_mode}")


def conductance_edges(Ei, Z, scenario, i, cfg=CFG):
    theta = Ei["theta_edge"].to_numpy(dtype=float)
    m = 1.0 / (1.0 + np.exp(-(float(Z) - theta) / max(cfg.activation_width_Z, 1e-12)))
    c_smooth = cfg.base_conductance + cfg.vertical_gain * m
    omega = omega_for_scenario(scenario, cfg)
    phases = edge_phases(Ei, scenario, i, cfg)

    if scenario.amplitude == 0:
        mod = np.ones_like(c_smooth)
    else:
        mod = 1.0 + float(scenario.amplitude) * np.cos(omega * float(Z) + phases)
    c_eff = np.clip(c_smooth * mod, 1e-9, None)

    Ec = Ei.copy()
    Ec["Z"] = float(Z)
    Ec["scenario"] = scenario.name
    Ec["scenario_amplitude"] = float(scenario.amplitude)
    Ec["scenario_omega"] = float(omega)
    Ec["scenario_phase_mode"] = scenario.phase_mode
    Ec["m_eff"] = m
    Ec["c_smooth"] = c_smooth
    Ec["modulation_factor"] = mod
    Ec["c_eff"] = c_eff
    return Ec


def weighted_laplacian_sparse(n, Ec):
    rows, cols, data = [], [], []
    deg = np.zeros(n, dtype=float)
    for _, e in Ec.iterrows():
        u, v, c = int(e["u"]), int(e["v"]), float(e["c_eff"])
        rows.extend([u, v])
        cols.extend([v, u])
        data.extend([-c, -c])
        deg[u] += c
        deg[v] += c
    rows.extend(range(n))
    cols.extend(range(n))
    data.extend(deg.tolist())
    return sp.csr_matrix((data, (rows, cols)), shape=(n, n))

## Stochastic heat trace

In [ ]:
# ============================================================
# Stochastic heat trace
# ============================================================

def rademacher_probes(n, n_probes, seed):
    rng = np.random.default_rng(seed)
    return rng.choice([-1.0, 1.0], size=(n, int(n_probes)))


def stochastic_heat_trace_expm(L, u_grid, probes):
    """Estimate normalized heat trace P(u)=Tr(exp(-uL))/N by Hutchinson."""
    n = L.shape[0]
    estimates = []
    # expm_multiply can evaluate a sequence of times in one call.
    # For each probe vector v, compute exp(-uL)v for all u.
    for k in range(probes.shape[1]):
        v = probes[:, k]
        Y = expm_multiply(-L, v, start=float(u_grid[0]), stop=float(u_grid[-1]), num=len(u_grid), endpoint=True)
        # Y shape: (len(u_grid), n)
        vals = np.einsum("i,ti->t", v, Y) / float(n)
        estimates.append(vals)
    P = np.mean(np.vstack(estimates), axis=0)
    return np.clip(P, 1e-300, 1.0)


def ds_curve_from_heat(u_grid, P):
    logu = np.log(u_grid)
    logP = np.log(np.clip(P, 1e-300, None))
    return -2.0 * np.gradient(logP, logu)


def ds_eff_from_curve(u_grid, P, ds_curve, n_points):
    p_low = max(5.0 / max(n_points, 1), 1e-4)
    mask = (P < 0.90) & (P > p_low) & np.isfinite(ds_curve) & (ds_curve > -1.0) & (ds_curve < 8.0)
    if int(mask.sum()) < 5:
        lo, hi = np.quantile(np.arange(len(u_grid)), [0.25, 0.75]).astype(int)
        mask = np.zeros_like(u_grid, dtype=bool)
        mask[lo:hi+1] = True
    vals = ds_curve[mask]
    return {
        "ds_eff_median": float(np.median(vals)),
        "ds_eff_mean": float(np.mean(vals)),
        "ds_eff_std": float(np.std(vals)),
        "window_points": int(mask.sum()),
        "u_window_min": float(u_grid[mask].min()),
        "u_window_max": float(u_grid[mask].max()),
    }


def compute_lambda1_scaled_sparse(L, i):
    try:
        vals = eigsh(L, k=3, which="SM", return_eigenvectors=False, tol=1e-5, maxiter=20000)
        vals = np.sort(np.real(vals))
        nz = vals[vals > 1e-8]
        lambda1 = float(nz[0]) if len(nz) else float("nan")
    except Exception:
        lambda1 = float("nan")
    return lambda1, float((4 ** int(i)) * lambda1) if np.isfinite(lambda1) else float("nan")


def scenario_cache_path(scenario_name, cfg=CFG):
    mode = "fast" if cfg.RUN_FAST else "publication"
    return CACHE_DIR / f"i{cfg.level_i}_{scenario_name}_{mode}_summary.csv"


def run_i5_stochastic_scan(cfg=CFG, scenarios=SCENARIOS):
    if not SCIPY_AVAILABLE:
        raise RuntimeError("scipy is required for sparse stochastic heat trace.")

    ap = active_params(cfg)
    n_Z, n_u, n_probes = ap["n_Z"], ap["n_u"], ap["n_probes"]

    i = int(cfg.level_i)
    n = num_points_i(i)
    Z_grid = np.linspace(cfg.Z_min, cfg.Z_max, n_Z)
    u_grid = np.logspace(math.log10(cfg.u_min), math.log10(cfg.u_max), n_u)
    u_mid = float(u_grid[len(u_grid)//2])
    Ei = prepare_edge_template(i, cfg)

    summary_all = []
    heat_rows = []

    for scenario in scenarios:
        cache_path = scenario_cache_path(scenario.name, cfg)
        if cache_path.exists():
            print("[cache] scenario:", scenario.name)
            df = pd.read_csv(cache_path)
            summary_all.append(df)
            continue

        print("[compute] scenario:", scenario.name)
        rows = []
        probes = rademacher_probes(n, n_probes, cfg.random_seed + len(scenario.name) + 97*i)
        for zi, Z in enumerate(Z_grid):
            Ec = conductance_edges(Ei, Z, scenario, i, cfg)
            L = weighted_laplacian_sparse(n, Ec)
            P = stochastic_heat_trace_expm(L, u_grid, probes)
            ds_curve = ds_curve_from_heat(u_grid, P)
            eff = ds_eff_from_curve(u_grid, P, ds_curve, n)
            lambda1, lambda1_scaled = compute_lambda1_scaled_sparse(L, i) if (zi % max(1, n_Z//8) == 0 or zi in [0, n_Z-1]) else (float("nan"), float("nan"))
            rows.append({
                "i": int(i),
                "scenario": scenario.name,
                "amplitude": float(scenario.amplitude),
                "omega_mode": scenario.omega_mode,
                "phase_mode": scenario.phase_mode,
                "Z": float(Z),
                "n_points": int(n),
                "num_edges": int(len(Ei)),
                "n_probes": int(n_probes),
                "m_eff_mean": float(Ec["m_eff"].mean()),
                "c_smooth_mean": float(Ec["c_smooth"].mean()),
                "modulation_factor_mean": float(Ec["modulation_factor"].mean()),
                "modulation_factor_std": float(Ec["modulation_factor"].std(ddof=0)),
                "c_eff_mean": float(Ec["c_eff"].mean()),
                "c_eff_std": float(Ec["c_eff"].std(ddof=0)),
                "lambda1_sparse_sampled": lambda1,
                "lambda1_scaled_4powi_sampled": lambda1_scaled,
                "u_mid": u_mid,
                "P_mid": float(P[len(u_grid)//2]),
                "logP_mid": float(np.log(P[len(u_grid)//2])),
                **eff,
            })
            for u, p, ds in zip(u_grid, P, ds_curve):
                heat_rows.append({
                    "i": int(i), "scenario": scenario.name, "Z": float(Z),
                    "u": float(u), "P_est": float(p), "logP_est": float(np.log(p)), "ds_curve": float(ds)
                })
        df = pd.DataFrame(rows)
        df.to_csv(cache_path, index=False)
        summary_all.append(df)

    summary = pd.concat(summary_all, ignore_index=True)
    heat = pd.DataFrame(heat_rows)
    summary.to_csv(OUT_DIR / "test6e_i5_stochastic_spectral_summary_by_Z.csv", index=False)
    heat.to_csv(OUT_DIR / "test6e_i5_stochastic_heat_trace_curves.csv", index=False)
    pd.DataFrame({"u": u_grid}).to_csv(OUT_DIR / "test6e_u_grid.csv", index=False)
    return summary, heat

## DSI residual analysis

In [ ]:
# ============================================================
# DSI residual analysis
# ============================================================

def fit_smooth_baseline(Z, y, degree=3):
    Z = np.asarray(Z, dtype=float)
    y = np.asarray(y, dtype=float)
    deg = min(int(degree), max(1, len(Z)-2))
    Zc = Z - np.mean(Z)
    coeff = np.polyfit(Zc, y, deg)
    yhat = np.polyval(coeff, Zc)
    rss = float(np.sum((y-yhat)**2))
    return yhat, {"baseline_model": f"poly{deg}", "baseline_params": [float(x) for x in coeff], "baseline_rss": rss}


def fit_fixed_frequency(Z, residual, omega):
    Z = np.asarray(Z, dtype=float)
    r = np.asarray(residual, dtype=float)
    X = np.column_stack([np.cos(omega*Z), np.sin(omega*Z), np.ones_like(Z)])
    beta, *_ = np.linalg.lstsq(X, r, rcond=None)
    pred = X @ beta
    rss = float(np.sum((r-pred)**2))
    amp = float(math.sqrt(beta[0]**2 + beta[1]**2))
    phase = float(math.atan2(-beta[1], beta[0]))
    return {"omega": float(omega), "amplitude": amp, "phase": phase, "rss": rss, "params": [float(x) for x in beta], "prediction": pred}


def aic_bic(rss, n, k):
    rss = max(float(rss), 1e-300)
    return float(n*math.log(rss/n)+2*k), float(n*math.log(rss/n)+k*math.log(n))


def permutation_pvalue(Z, residual, omega, observed_amp, n_perm, seed):
    rng = np.random.default_rng(seed)
    r = np.asarray(residual, dtype=float)
    amps = []
    for _ in range(int(n_perm)):
        rp = rng.permutation(r)
        amps.append(fit_fixed_frequency(Z, rp, omega)["amplitude"])
    amps = np.asarray(amps)
    return float((np.sum(amps >= observed_amp)+1)/(len(amps)+1)), float(np.mean(amps)), float(np.std(amps))


def frequency_scan(Z, residual, omega_min, omega_max, n_grid=500):
    rows = []
    if omega_max <= omega_min:
        return pd.DataFrame()
    for omega in np.linspace(omega_min, omega_max, n_grid):
        fit = fit_fixed_frequency(Z, residual, float(omega))
        rows.append({k:v for k,v in fit.items() if k != "prediction"})
    return pd.DataFrame(rows).sort_values("rss").reset_index(drop=True)


def analyze_observable(summary, scenario, observable, cfg=CFG):
    omega0 = omega0_lattice(cfg)
    ap = active_params(cfg)
    sub = summary[summary["scenario"] == scenario].sort_values("Z")
    Z = sub["Z"].to_numpy(dtype=float)
    y = sub[observable].to_numpy(dtype=float)
    dZ = float(np.median(np.diff(Z)))
    omega_nyquist = float(math.pi/dZ)
    resolvable = bool(omega0 < 0.8*omega_nyquist)

    baseline, base_info = fit_smooth_baseline(Z, y, cfg.baseline_poly_degree)
    residual = y - baseline
    residual -= np.mean(residual)
    fixed = fit_fixed_frequency(Z, residual, omega0)
    rss_null = float(np.sum(residual**2))
    aic_null, bic_null = aic_bic(rss_null, len(Z), 1)
    aic_fixed, bic_fixed = aic_bic(fixed["rss"], len(Z), 3)
    p_perm, null_amp_mean, null_amp_std = permutation_pvalue(
        Z, residual, omega0, fixed["amplitude"],
        ap["permutation_count"], cfg.random_seed + len(scenario) + len(observable)
    )
    scan_max = min(40.0, 0.95*omega_nyquist)
    scan = frequency_scan(Z, residual, 0.5, scan_max, n_grid=500)
    best_omega = float(scan.iloc[0]["omega"]) if len(scan) else float("nan")
    best_rss = float(scan.iloc[0]["rss"]) if len(scan) else float("nan")
    omega0_rank = int(np.argmin(np.abs(scan["omega"].to_numpy() - omega0))+1) if len(scan) else None

    if not resolvable:
        verdict = "not_resolvable"
    elif (aic_null - aic_fixed > 4.0) and (p_perm < 0.05):
        verdict = "candidate"
    else:
        verdict = "not_detected"

    row = {
        "i": int(cfg.level_i), "scenario": scenario, "observable": observable,
        "n_Z": int(len(Z)), "dZ_median": dZ,
        "omega0_lattice": omega0, "omega_nyquist": omega_nyquist, "omega0_resolvable": resolvable,
        "baseline_model": base_info["baseline_model"], "baseline_rss": float(base_info["baseline_rss"]),
        "residual_rss_null": rss_null,
        "fixed_omega_amplitude": fixed["amplitude"], "fixed_omega_phase": fixed["phase"], "fixed_omega_rss": fixed["rss"],
        "delta_aic_null_minus_fixed": float(aic_null - aic_fixed),
        "delta_bic_null_minus_fixed": float(bic_null - bic_fixed),
        "permutation_p_value": p_perm,
        "permutation_amp_mean": null_amp_mean, "permutation_amp_std": null_amp_std,
        "best_scan_omega": best_omega, "best_scan_rss": best_rss,
        "omega_distance_to_lattice": float(abs(best_omega - omega0)) if np.isfinite(best_omega) else float("nan"),
        "omega0_rank_nearest": omega0_rank,
        "curve_verdict": verdict,
    }
    curve = pd.DataFrame({
        "i": int(cfg.level_i), "scenario": scenario, "observable": observable,
        "Z": Z, "y": y, "baseline": baseline, "residual": residual, "fixed_omega_fit": fixed["prediction"]
    })
    scan["i"] = int(cfg.level_i); scan["scenario"] = scenario; scan["observable"] = observable
    return row, curve, scan


def run_dsi_analysis(summary, cfg=CFG):
    observables = ["c_eff_mean", "modulation_factor_mean", "ds_eff_median", "ds_eff_mean", "logP_mid"]
    rows, curves, scans = [], [], []
    for scenario in sorted(summary["scenario"].unique()):
        for obs in observables:
            row, curve, scan = analyze_observable(summary, scenario, obs, cfg)
            rows.append(row); curves.append(curve); scans.append(scan)

    results = pd.DataFrame(rows)
    curves_df = pd.concat(curves, ignore_index=True)
    scans_df = pd.concat(scans, ignore_index=True)

    results.to_csv(OUT_DIR / "test6e_i5_DSI_results_by_curve.csv", index=False)
    curves_df.to_csv(OUT_DIR / "test6e_i5_DSI_residual_curves.csv", index=False)
    scans_df.to_csv(OUT_DIR / "test6e_i5_DSI_frequency_scan.csv", index=False)

    scenario_summary = []
    for scenario, g in results.groupby("scenario"):
        obs_spectral = g[g["observable"].isin(["ds_eff_median", "ds_eff_mean", "logP_mid"])]
        candidate_rows = int((g["curve_verdict"] == "candidate").sum())
        spectral_candidates = int((obs_spectral["curve_verdict"] == "candidate").sum())
        scenario_summary.append({
            "i": int(cfg.level_i),
            "scenario": scenario,
            "total_rows": int(len(g)),
            "candidate_rows": candidate_rows,
            "spectral_rows": int(len(obs_spectral)),
            "spectral_candidate_rows": spectral_candidates,
            "scenario_verdict": "transmitted_candidate" if spectral_candidates >= 2 else ("conductance_only_candidate" if candidate_rows > spectral_candidates else "not_transmitted"),
        })
    scenario_summary = pd.DataFrame(scenario_summary)
    scenario_summary.to_csv(OUT_DIR / "test6e_i5_scenario_summary.csv", index=False)

    h1 = scenario_summary[scenario_summary["scenario"] == "H1_lattice_coherent_A002"]
    h0 = scenario_summary[scenario_summary["scenario"] == "H0_smooth"]
    nonlat = scenario_summary[scenario_summary["scenario"] == "CTRL_nonlattice_coherent_A002"]
    randomp = scenario_summary[scenario_summary["scenario"] == "CTRL_lattice_randomphase_A002"]

    h1_trans = bool(len(h1) and h1.iloc[0]["scenario_verdict"] == "transmitted_candidate")
    controls_rejected = bool(
        (len(h0)==0 or h0.iloc[0]["scenario_verdict"] != "transmitted_candidate") and
        (len(nonlat)==0 or nonlat.iloc[0]["scenario_verdict"] != "transmitted_candidate") and
        (len(randomp)==0 or randomp.iloc[0]["scenario_verdict"] != "transmitted_candidate")
    )

    if h1_trans and controls_rejected:
        global_verdict = "i5_DSI_transmission_candidate_controls_rejected"
    elif h1_trans:
        global_verdict = "i5_partial_transmission_candidate"
    else:
        global_verdict = "i5_not_transmitted"

    report = {
        "status": "completed",
        "not_a_DSI_proof": True,
        "global_verdict": global_verdict,
        "level_i": int(cfg.level_i),
        "omega0_lattice": omega0_lattice(cfg),
        "active_params": active_params(cfg),
        "scenario_summary": scenario_summary.to_dict(orient="records"),
        "interpretation": (
            "A positive i=5 result means that an explicitly imposed DSI modulation at conductance-link level "
            "is still detectable in stochastic heat-trace/spectral observables. It does not prove spontaneous DSI."
        ),
    }
    with open(OUT_DIR / "test6e_i5_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    return results, curves_df, scans_df, scenario_summary, report

## Figures and export

In [ ]:
# ============================================================
# Figures and export
# ============================================================

def make_figures(summary, curves_df, scenario_summary, cfg=CFG):
    fig_dir = OUT_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    for obs in ["c_eff_mean", "ds_eff_median", "logP_mid"]:
        plt.figure(figsize=(9,5))
        for scenario, g in summary.groupby("scenario"):
            plt.plot(g["Z"], g[obs], label=scenario)
        plt.xlabel("Z"); plt.ylabel(obs)
        plt.title(f"Test6e i=5 stochastic observable — {obs}")
        plt.grid(True, alpha=0.3); plt.legend(fontsize=8); plt.tight_layout()
        plt.savefig(fig_dir / f"observable_{obs}_i5.png", dpi=160)
        plt.close()

    for obs in ["ds_eff_median", "logP_mid"]:
        plt.figure(figsize=(9,5))
        sub = curves_df[curves_df["observable"] == obs]
        for scenario, g in sub.groupby("scenario"):
            plt.plot(g["Z"], g["residual"], label=scenario)
        plt.axhline(0, linewidth=1)
        plt.xlabel("Z"); plt.ylabel("residual")
        plt.title(f"Test6e i=5 residual — {obs}")
        plt.grid(True, alpha=0.3); plt.legend(fontsize=8); plt.tight_layout()
        plt.savefig(fig_dir / f"residual_{obs}_i5.png", dpi=160)
        plt.close()

    plt.figure(figsize=(8,5))
    x = np.arange(len(scenario_summary))
    plt.bar(x, scenario_summary["spectral_candidate_rows"])
    plt.xticks(x, scenario_summary["scenario"], rotation=45, ha="right", fontsize=8)
    plt.ylabel("spectral candidate rows")
    plt.title("Test6e i=5 stochastic spectral transmission")
    plt.tight_layout()
    plt.savefig(fig_dir / "scenario_spectral_candidate_rows_i5.png", dpi=160)
    plt.close()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()


def write_readme_manifest_and_zip():
    lines = [
        "# ROSG Test6e — i=5 stochastic heat-trace DSI transmission audit",
        "",
        "Purpose:",
        "- extend Test6d V2 from i=3,4 to i=5;",
        "- use stochastic Hutchinson trace estimation with expm_multiply;",
        "- compare H0, H1 lattice coherent A=0.02, non-lattice, and random-phase controls;",
        "- determine whether the DSI-modulated conductance transmission persists at i=5.",
        "",
        "Important:",
        "This does not prove spontaneous DSI.",
        "It tests whether an explicitly imposed link-level DSI modulation survives the heat-trace/spectral pipeline at i=5.",
        "",
        "Main files:",
        "- test6e_i5_stochastic_spectral_summary_by_Z.csv",
        "- test6e_i5_stochastic_heat_trace_curves.csv",
        "- test6e_i5_DSI_results_by_curve.csv",
        "- test6e_i5_scenario_summary.csv",
        "- test6e_i5_report.json",
        "- figures/",
        "- manifest_sha256.csv",
    ]
    (OUT_DIR / "README.md").write_text("\n".join(lines), encoding="utf-8")
    rows = []
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file() and p.name != "manifest_sha256.csv":
            rows.append({"path": str(p.relative_to(OUT_DIR)), "size_bytes": int(p.stat().st_size), "sha256": sha256_file(p)})
    pd.DataFrame(rows).to_csv(OUT_DIR / "manifest_sha256.csv", index=False)
    zip_path = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    print("ZIP:", zip_path)
    return zip_path

## Autotests and full audit

In [ ]:
def run_autotests():
    print("Running Test6e autotests...")
    if not SCIPY_AVAILABLE:
        raise RuntimeError("scipy is required.")
    ap = active_params(CFG)
    dZ = (CFG.Z_max-CFG.Z_min)/(ap["n_Z"]-1)
    assert omega0_lattice(CFG) < 0.8*(math.pi/dZ)
    assert num_points_i(CFG.level_i) == 1089
    Ei = prepare_edge_template(CFG.level_i, CFG)
    Ec = conductance_edges(Ei, 0.0, SCENARIOS[1], CFG.level_i, CFG)
    L = weighted_laplacian_sparse(num_points_i(CFG.level_i), Ec)
    assert L.shape == (1089, 1089)
    assert L.nnz > 0
    print("All autotests passed.")


def run_full_audit(cfg=CFG):
    run_autotests()
    summary, heat = run_i5_stochastic_scan(cfg, SCENARIOS)
    results, curves_df, scans_df, scenario_summary, report = run_dsi_analysis(summary, cfg)
    make_figures(summary, curves_df, scenario_summary, cfg)
    zip_path = write_readme_manifest_and_zip()
    return summary, heat, results, curves_df, scans_df, scenario_summary, report, zip_path

## Run Test6e

In [ ]:
summary, heat, results, curves_df, scans_df, scenario_summary, report, zip_path = run_full_audit(CFG)
print(json.dumps({
    "status": "completed",
    "zip_path": zip_path,
    "global_verdict": report["global_verdict"],
    "level_i": report["level_i"],
    "omega0_lattice": report["omega0_lattice"],
    "active_params": report["active_params"],
}, indent=2))
display(scenario_summary)
display(pd.DataFrame([report]))


Running Test6e autotests...
All autotests passed.
[compute] scenario: H0_smooth
[compute] scenario: H1_lattice_coherent_A002
[compute] scenario: CTRL_nonlattice_coherent_A002
[compute] scenario: CTRL_lattice_randomphase_A002
ZIP: /content/drive/MyDrive/ROSG_exports/ROSG_Test6e_i5_stochastic_heat_trace_DSI_transmission_audit.zip
{
  "status": "completed",
  "zip_path": "/content/drive/MyDrive/ROSG_exports/ROSG_Test6e_i5_stochastic_heat_trace_DSI_transmission_audit.zip",
  "global_verdict": "i5_DSI_transmission_candidate_controls_rejected",
  "level_i": 5,
  "omega0_lattice": 9.064720283654388,
  "active_params": {
    "n_Z": 65,
    "n_u": 34,
    "n_probes": 24,
    "permutation_count": 600
  }
}


,i,scenario,total_rows,candidate_rows,spectral_rows,spectral_candidate_rows,scenario_verdict
0,5,CTRL_lattice_randomphase_A002,5,1,3,0,conductance_only_candidate
1,5,CTRL_nonlattice_coherent_A002,5,0,3,0,not_transmitted
2,5,H0_smooth,5,0,3,0,not_transmitted
3,5,H1_lattice_coherent_A002,5,5,3,3,transmitted_candidate


,status,not_a_DSI_proof,global_verdict,level_i,omega0_lattice,active_params,scenario_summary,interpretation
0,completed,True,i5_DSI_transmission_candidate_controls_rejected,5,9.06472,"{'n_Z': 65, 'n_u': 34, 'n_probes': 24, 'permut...","[{'i': 5, 'scenario': 'CTRL_lattice_randomphas...",A positive i=5 result means that an explicitly...
